In [ ]:
from collections import namedtuple
from pathlib import Path
import os
import platform

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.stats import norm

CORE_VERSION = "2026_10_27_simulation_core_v2"

RUN_MODE = os.environ.get("SIM_RUN_MODE", "production").strip().lower()
if RUN_MODE not in ("quick", "production"):
    raise ValueError("SIM_RUN_MODE must be quick or production")

MAX_STEP_HOURS = 1.0 if RUN_MODE == "quick" else 0.5
OUTPUT_STEP_HOURS = 4.0 if RUN_MODE == "quick" else 1.0
ATOL = 1e-11
RTOL = 1e-6
NEGATIVE_STATE_TOLERANCE = -1e-8

TOTAL_DENSITY_SURVIVAL_THRESHOLD = 0.1
SUBPOPULATION_CUTOFF = 0.0

COMPOSITION_CONVERGENCE_TOLERANCE = 0.001
BOUNDARY_DENSITY_LOG_CONVERGENCE_TOLERANCE = 0.01
ENDPOINT_TCN_LOG_CONVERGENCE_TOLERANCE = 0.001
CONSECUTIVE_CONVERGED_CYCLES = 8
MIN_CONVERGENCE_CYCLES = 60
MAX_CONVERGENCE_CYCLES = 120 if RUN_MODE == "quick" else 250
ADDITIONAL_CYCLES_AFTER_CONVERGENCE = 3

PASSAGE_INTERVAL_HOURS = 24.0
PASSAGE_DILUTION_FACTOR = 30000.0
PASSAGE_GENERATIONS = float(np.log2(PASSAGE_DILUTION_FACTOR))
RELAXED_DAYS_PER_CYCLE = 5
SELECTED_DAYS_PER_CYCLE = 2
DAYS_PER_CYCLE = RELAXED_DAYS_PER_CYCLE + SELECTED_DAYS_PER_CYCLE
TRANSIENT_CYCLES = 5

PERIODIC_INITIAL_MEAN_FRACTION_OF_PCN = 0.50
PERIODIC_INITIAL_SD_FRACTION_OF_PCN = 0.30
FIXED_LOSS_INITIAL_MEAN_FRACTION_OF_PCN = 0.80
FIXED_GAIN_INITIAL_MEAN_FRACTION_OF_PCN = 0.10
FIXED_INITIAL_SD_FRACTION_OF_PCN = 0.15

ModelParameters = namedtuple(
    "ModelParameters",
    "mu_max mu_infinity burden_scale delta_antibiotic gamma alpha steepness "
    "antibiotic_relaxed antibiotic_selected carrying_capacity",
)
MODEL = ModelParameters(2.0, 0.8, 35.0, 2.0, 0.002, 40.0, 2.0, 0.0, 2.0, 1.0)

TransitionParameters = namedtuple(
    "TransitionParameters",
    "kappa_f_off kappa_b_off kappa_f_on kappa_b_on kappa_f_pcn kappa_b_pcn lower_forward_rate",
)
TRANSITIONS = TransitionParameters(0.0075, 0.030, 0.100, 0.120, 0.250, 0.220, 0.0075)

GrowthProfile = namedtuple("GrowthProfile", "name burden_scale protection_alpha")

TransitionConfiguration = namedtuple(
    "TransitionConfiguration",
    "condition max_pcn setting_label kappa_f kappa_b lower_forward_rate",
)

CONDITION_ORDER = (
    "transposition off",
    "transposition on",
    "transposition on + PCN dynamics",
)
DISPLAY_LABELS = dict(zip(CONDITION_ORDER, (
    "baseline",
    "+ transposition",
    "+ transposition + PCN dynamics",
)))
CONDITION_COLORS = dict(zip(CONDITION_ORDER, ("#C3C5DE", "#8C82BC", "#3A2A72")))
CONDITION_LINEWIDTHS = dict(zip(CONDITION_ORDER, (1.1, 1.4, 1.7)))
SELECTION_SHADE = "#DDF1D8"

PLOT_STYLE = dict(zip(
    (
        "font.family", "font.sans-serif", "font.size", "axes.labelsize", "axes.titlesize",
        "axes.linewidth", "axes.spines.top", "axes.spines.right", "xtick.labelsize",
        "ytick.labelsize", "legend.fontsize", "lines.solid_capstyle", "figure.dpi",
        "savefig.dpi", "pdf.fonttype", "ps.fonttype",
    ),
    (
        "sans-serif", ["Arial", "Nimbus Sans", "DejaVu Sans"], 8, 8, 8,
        0.8, False, False, 7,
        7, 7, "round", 110,
        300, 42, 42,
    ),
))


def total_tcn_states(max_pcn, chromosomal_copy=1):
    if int(max_pcn) != max_pcn or max_pcn < 1:
        raise ValueError("max_pcn must be a positive integer")
    if chromosomal_copy not in (0, 1):
        raise ValueError("chromosomal_copy must be 0 or 1")
    return np.arange(chromosomal_copy, int(max_pcn) + chromosomal_copy + 1, dtype=float)


def growth_rates(max_pcn, selected, chromosomal_copy=1, model=MODEL, growth_profile=None):
    total_tcn = total_tcn_states(max_pcn, chromosomal_copy)
    plasmid_tcn = total_tcn - chromosomal_copy
    profile = growth_profile or GrowthProfile("default", model.burden_scale, model.alpha)
    relaxed = model.mu_infinity + (model.mu_max - model.mu_infinity) / (
        1.0 + plasmid_tcn / profile.burden_scale
    )
    antibiotic = model.antibiotic_selected if selected else model.antibiotic_relaxed
    half_effect = model.gamma * (1.0 + profile.protection_alpha * total_tcn)
    inhibition = model.delta_antibiotic * antibiotic**model.steepness / (
        antibiotic**model.steepness + half_effect**model.steepness
    )
    return relaxed - inhibition


def make_configuration(max_pcn, condition, kappa_f, kappa_b, setting_label, lower_forward_rate=None):
    if condition not in CONDITION_ORDER:
        raise ValueError("unknown condition " + str(condition))
    if kappa_f <= 0 or kappa_b <= 0:
        raise ValueError("kappa_f and kappa_b must be positive")
    if lower_forward_rate is None:
        lower_forward_rate = TRANSITIONS.lower_forward_rate
    return TransitionConfiguration(
        condition, int(max_pcn), setting_label,
        float(kappa_f), float(kappa_b), float(lower_forward_rate),
    )


def standard_configurations(max_pcn, setting_label="Figure 2 primary"):
    pairs = (
        (TRANSITIONS.kappa_f_off, TRANSITIONS.kappa_b_off),
        (TRANSITIONS.kappa_f_on, TRANSITIONS.kappa_b_on),
        (TRANSITIONS.kappa_f_pcn, TRANSITIONS.kappa_b_pcn),
    )
    return [
        make_configuration(max_pcn, condition, kf, kb, setting_label)
        for condition, (kf, kb) in zip(CONDITION_ORDER, pairs)
    ]


def transition_matrix(max_pcn, config):
    if int(max_pcn) != config.max_pcn:
        raise ValueError("configuration and matrix use different max_pcn")
    n_states = int(max_pcn) + 1
    matrix = np.zeros((n_states, n_states))
    for source in range(n_states):
        if source == 0:
            forward = config.lower_forward_rate
        elif source < max_pcn:
            forward = config.kappa_f
        else:
            forward = 0.0
        backward = config.kappa_b if source > 0 else 0.0
        if forward > 0:
            matrix[source + 1, source] += forward
            matrix[source, source] -= forward
        if backward > 0:
            matrix[source - 1, source] += backward
            matrix[source, source] -= backward
    return matrix


def assert_generator_valid(matrix, label=""):
    if not np.allclose(matrix.sum(axis=0), 0.0, atol=1e-12):
        raise RuntimeError("transition matrix does not conserve density " + label)
    off_diagonal = matrix - np.diag(np.diag(matrix))
    if (off_diagonal < -1e-14).any():
        raise RuntimeError("transition matrix has a negative off diagonal entry " + label)


def normalize_state(state, total_abundance=None):
    if total_abundance is None:
        total_abundance = MODEL.carrying_capacity / PASSAGE_DILUTION_FACTOR
    state = np.asarray(state, dtype=float)
    if np.any(state < 0) or not np.isfinite(state).all() or state.sum() <= 0:
        raise ValueError("state weights must be finite, nonnegative, and positive in total")
    return state * (float(total_abundance) / float(state.sum()))


def normal_initial_state(max_pcn, mean_fraction_of_pcn, sd_fraction_of_pcn, chromosomal_copy=1):
    states = total_tcn_states(max_pcn, chromosomal_copy)
    weights = norm.pdf(
        states, loc=mean_fraction_of_pcn * max_pcn, scale=sd_fraction_of_pcn * max_pcn
    )
    return normalize_state(weights)


def periodic_initial_state(max_pcn, chromosomal_copy=1):
    return normal_initial_state(
        max_pcn, PERIODIC_INITIAL_MEAN_FRACTION_OF_PCN,
        PERIODIC_INITIAL_SD_FRACTION_OF_PCN, chromosomal_copy,
    )


def fixed_loss_initial_state(max_pcn, chromosomal_copy=1):
    return normal_initial_state(
        max_pcn, FIXED_LOSS_INITIAL_MEAN_FRACTION_OF_PCN,
        FIXED_INITIAL_SD_FRACTION_OF_PCN, chromosomal_copy,
    )


def fixed_gain_initial_state(max_pcn, chromosomal_copy=1):
    return normal_initial_state(
        max_pcn, FIXED_GAIN_INITIAL_MEAN_FRACTION_OF_PCN,
        FIXED_INITIAL_SD_FRACTION_OF_PCN, chromosomal_copy,
    )


def uniform_initial_state(max_pcn):
    return normalize_state(np.ones(int(max_pcn) + 1))


def random_initial_state(max_pcn, rng):
    return normalize_state(rng.uniform(0.0, 1.0, size=int(max_pcn) + 1))


def normalized_composition(state):
    state = np.maximum(np.asarray(state, dtype=float), 0.0)
    total = float(state.sum())
    return state / total if total > 0 else np.full_like(state, np.nan)


def check_negative(values):
    lowest = float(np.min(values))
    if lowest < NEGATIVE_STATE_TOLERANCE:
        raise FloatingPointError("state below negative tolerance " + str(lowest))
    return lowest


def summarize_state(state, growth, chromosomal_copy=1):
    lowest = check_negative(state)
    state = np.maximum(np.asarray(state, dtype=float), 0.0)
    density = float(state.sum())
    tcn_signal = float(np.dot(state, total_tcn_states(len(state) - 1, chromosomal_copy)))
    return dict(
        density=density,
        TCN_signal=tcn_signal,
        TCN_per_density=tcn_signal / density if density > 0 else np.nan,
        mean_intrinsic_growth=float(np.dot(state / density, growth)) if density > 0 else np.nan,
        minimum_raw_state=lowest,
    )


def evaluation_times():
    times = np.arange(0.0, PASSAGE_INTERVAL_HOURS + OUTPUT_STEP_HOURS, OUTPUT_STEP_HOURS)
    times = times[times <= PASSAGE_INTERVAL_HOURS]
    if not np.isclose(times[-1], PASSAGE_INTERVAL_HOURS):
        times = np.append(times, PASSAGE_INTERVAL_HOURS)
    return times


def ode_rhs(_time, state, growth, transitions, carrying_capacity):
    return (growth * state + transitions @ state) * (1.0 - np.sum(state) / carrying_capacity)


def integrate_day(state, growth, transitions, record_trajectory):
    t_eval = evaluation_times() if record_trajectory else np.array([PASSAGE_INTERVAL_HOURS])
    solution = solve_ivp(
        ode_rhs, (0.0, PASSAGE_INTERVAL_HOURS), np.asarray(state, dtype=float),
        args=(growth, transitions, MODEL.carrying_capacity),
        method="LSODA", t_eval=t_eval, max_step=MAX_STEP_HOURS, atol=ATOL, rtol=RTOL,
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    if not np.isfinite(solution.y).all():
        raise FloatingPointError("non finite state")
    check_negative(solution.y)
    return solution.t, solution.y, np.maximum(solution.y[:, -1], 0.0)


def population_is_extinct(state):
    return float(np.sum(np.maximum(state, 0.0))) <= 0.0


def simulate_phase(max_pcn, config, phase, duration_days, initial_state, record_trajectory,
                   chromosomal_copy=1, growth_profile=None):
    if phase not in ("loss", "gain"):
        raise ValueError("phase must be loss or gain")
    growth = growth_rates(max_pcn, phase == "gain", chromosomal_copy, growth_profile=growth_profile)
    transitions = transition_matrix(max_pcn, config)
    state = np.asarray(initial_state, dtype=float).copy()
    frames, daily_rows = [], []
    start = summarize_state(state, growth, chromosomal_copy)
    last = start

    for day in range(duration_days):
        if population_is_extinct(state):
            daily_rows.append(dict(phase_day=day + 1, prepassage_density=0.0,
                                   prepassage_TCN=np.nan, prepassage_TCN_signal=0.0))
            break
        times, states, prepassage = integrate_day(state, growth, transitions, record_trajectory)
        last = summarize_state(prepassage, growth, chromosomal_copy)
        daily_rows.append(dict(phase_day=day + 1, prepassage_density=last["density"],
                               prepassage_TCN=last["TCN_per_density"],
                               prepassage_TCN_signal=last["TCN_signal"]))

        if record_trajectory:
            frame = pd.DataFrame([
                dict(phase_day=day + hour / PASSAGE_INTERVAL_HOURS,
                     phase_time_hours=day * PASSAGE_INTERVAL_HOURS + hour,
                     period=day + 1, passage_status="within_period",
                     **summarize_state(states[:, k], growth, chromosomal_copy))
                for k, hour in enumerate(times)
            ])
            frame.loc[frame.index[-1], "passage_status"] = "pre_passage"
            if day == 0:
                frame.loc[frame.index[0], "passage_status"] = "phase_start"
            else:
                frame = frame.iloc[1:].copy()
            frames.append(frame)

        state = prepassage / PASSAGE_DILUTION_FACTOR
        if SUBPOPULATION_CUTOFF > 0:
            state[(state > 0) & (state < SUBPOPULATION_CUTOFF)] = 0.0

        if record_trajectory:
            frames.append(pd.DataFrame([dict(
                phase_day=day + 1.0, phase_time_hours=(day + 1) * PASSAGE_INTERVAL_HOURS,
                period=day + 1, passage_status="post_passage",
                **summarize_state(state, growth, chromosomal_copy),
            )]))

    trajectory = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    daily = pd.DataFrame(daily_rows)
    for frame in (trajectory, daily):
        if not frame.empty:
            frame["phase"] = phase
            frame["condition"] = config.condition
            frame["maximum_PCN"] = int(max_pcn)
            frame["setting_label"] = config.setting_label
            frame["chromosomal_copy"] = chromosomal_copy

    summary = dict(
        phase=phase, condition=config.condition, maximum_PCN=int(max_pcn),
        setting_label=config.setting_label, chromosomal_copy=chromosomal_copy,
        initial_TCN=float(start["TCN_per_density"]),
        final_TCN=float(last["TCN_per_density"]),
        final_prepassage_density=float(last["density"]),
        minimum_daily_prepassage_density=float(daily["prepassage_density"].min()) if not daily.empty else 0.0,
        postphase_boundary_density=float(np.sum(state)),
        kappa_f=config.kappa_f, kappa_b=config.kappa_b,
        lower_forward_rate=config.lower_forward_rate,
        population_extinct=population_is_extinct(state),
    )
    return trajectory, daily, state, summary


def run_cycle(max_pcn, config, state, record_trajectory, chromosomal_copy=1, growth_profile=None):
    loss_traj, loss_daily, after_loss, loss = simulate_phase(
        max_pcn, config, "loss", RELAXED_DAYS_PER_CYCLE, state, record_trajectory,
        chromosomal_copy, growth_profile,
    )
    gain_traj, gain_daily, after_gain, gain = simulate_phase(
        max_pcn, config, "gain", SELECTED_DAYS_PER_CYCLE, after_loss, record_trajectory,
        chromosomal_copy, growth_profile,
    )
    return (loss_traj, loss_daily, loss), (gain_traj, gain_daily, gain), after_gain


def label_cycle(frame, cycle, phase):
    offset = (cycle - 1) * DAYS_PER_CYCLE + (RELAXED_DAYS_PER_CYCLE if phase == "gain" else 0)
    frame = frame.copy()
    frame["cycle"] = cycle
    frame["phase"] = phase
    frame["Day"] = offset + frame["phase_day"]
    return frame


def simulate_transient_cycles(max_pcn, config, cycles=TRANSIENT_CYCLES, initial_state=None,
                              chromosomal_copy=1, growth_profile=None):
    if initial_state is None:
        state = periodic_initial_state(max_pcn, chromosomal_copy)
    else:
        state = np.asarray(initial_state, dtype=float).copy()
    trajectories, dailies, rows = [], [], []
    for cycle in range(1, cycles + 1):
        (lt, ld, loss), (gt, gd, gain), state = run_cycle(
            max_pcn, config, state, True, chromosomal_copy, growth_profile
        )
        trajectories += [label_cycle(lt, cycle, "loss"), label_cycle(gt, cycle, "gain")]
        dailies += [label_cycle(ld, cycle, "loss"), label_cycle(gd, cycle, "gain")]
        rows.append(dict(
            cycle=cycle, condition=config.condition, maximum_PCN=int(max_pcn),
            setting_label=config.setting_label, chromosomal_copy=chromosomal_copy,
            TCN_low_end_relaxed=loss["final_TCN"], TCN_high_end_selected=gain["final_TCN"],
            density_end_relaxed=loss["final_prepassage_density"],
            density_end_selected=gain["final_prepassage_density"],
            minimum_daily_prepassage_density=min(loss["minimum_daily_prepassage_density"],
                                                 gain["minimum_daily_prepassage_density"]),
        ))
    return (pd.concat(trajectories, ignore_index=True),
            pd.concat(dailies, ignore_index=True),
            pd.DataFrame(rows))


def log_difference(current, previous):
    if previous is None or not np.isfinite(current) or not np.isfinite(previous):
        return np.inf
    if current <= 0 or previous <= 0:
        return np.inf
    return float(abs(np.log(current / previous)))


def converge_periodic_state(max_pcn, config, initial_state, chromosomal_copy=1, growth_profile=None):
    state = np.asarray(initial_state, dtype=float).copy()
    previous_composition = normalized_composition(state)
    previous_density = float(state.sum())
    previous_low = previous_high = None
    stable = 0

    for cycle in range(1, MAX_CONVERGENCE_CYCLES + 1):
        (_, _, loss), (_, _, gain), state = run_cycle(
            max_pcn, config, state, False, chromosomal_copy, growth_profile
        )
        extinct = bool(loss["population_extinct"] or gain["population_extinct"])
        low, high = float(loss["final_TCN"]), float(gain["final_TCN"])
        density = float(state.sum())
        composition = normalized_composition(state)
        minimum_density = min(float(loss["minimum_daily_prepassage_density"]),
                              float(gain["minimum_daily_prepassage_density"]))

        if extinct or not np.isfinite(composition).all():
            composition_error = np.inf
        else:
            composition_error = float(np.sum(np.abs(composition - previous_composition)))
        criteria_met = (
            not extinct
            and composition_error < COMPOSITION_CONVERGENCE_TOLERANCE
            and log_difference(density, previous_density) < BOUNDARY_DENSITY_LOG_CONVERGENCE_TOLERANCE
            and max(log_difference(low, previous_low), log_difference(high, previous_high))
            < ENDPOINT_TCN_LOG_CONVERGENCE_TOLERANCE
        )
        stable = stable + 1 if criteria_met else 0

        result = dict(
            converged=False, population_extinct=extinct, state=state,
            cycles_to_convergence=cycle, TCN_low=low, TCN_high=high,
            minimum_daily_prepassage_density=minimum_density,
        )
        if extinct:
            return result
        if cycle >= MIN_CONVERGENCE_CYCLES and stable >= CONSECUTIVE_CONVERGED_CYCLES:
            result["converged"] = True
            return result

        previous_composition, previous_density = composition, density
        previous_low, previous_high = low, high

    return result


def periodic_metrics(low_tcn, high_tcn, minimum_daily_prepassage_density, converged, population_extinct):
    survived = bool(np.isfinite(minimum_daily_prepassage_density)
                    and minimum_daily_prepassage_density >= TOTAL_DENSITY_SURVIVAL_THRESHOLD)
    valid = bool(converged and not population_extinct and survived
                 and np.isfinite(low_tcn) and np.isfinite(high_tcn)
                 and low_tcn > 0 and high_tcn > 0)
    if not valid:
        return dict(cycle_amplitude=np.nan, loss_rate=np.nan, gain_rate=np.nan,
                    survived_protocol=survived, response_valid=False)
    amplitude = float(np.log(high_tcn / low_tcn))
    return dict(
        cycle_amplitude=amplitude,
        loss_rate=amplitude / (RELAXED_DAYS_PER_CYCLE * PASSAGE_GENERATIONS),
        gain_rate=amplitude / (SELECTED_DAYS_PER_CYCLE * PASSAGE_GENERATIONS),
        survived_protocol=True,
        response_valid=True,
    )


def evaluate_configuration(max_pcn, config, initial_state=None, chromosomal_copy=1, growth_profile=None):
    if initial_state is None:
        initial_state = periodic_initial_state(max_pcn, chromosomal_copy)
    result = converge_periodic_state(max_pcn, config, initial_state, chromosomal_copy, growth_profile)
    metrics = periodic_metrics(
        result["TCN_low"], result["TCN_high"], result["minimum_daily_prepassage_density"],
        result["converged"], result["population_extinct"],
    )
    valid = metrics["response_valid"]
    return dict(
        condition=config.condition, setting_label=config.setting_label,
        maximum_PCN=int(max_pcn), chromosomal_copy=chromosomal_copy,
        kappa_f=config.kappa_f, kappa_b=config.kappa_b,
        converged=result["converged"], population_extinct=result["population_extinct"],
        cycles_to_convergence=result["cycles_to_convergence"],
        TCN_low=result["TCN_low"] if valid else np.nan,
        TCN_high=result["TCN_high"] if valid else np.nan,
        minimum_daily_prepassage_density=result["minimum_daily_prepassage_density"],
        **metrics,
    )


def simulate_post_convergence_cycles(max_pcn, config, converged_state, chromosomal_copy=1,
                                     growth_profile=None):
    state = np.asarray(converged_state, dtype=float).copy()
    frames = []
    for cycle in range(1, ADDITIONAL_CYCLES_AFTER_CONVERGENCE + 1):
        (lt, _, _), (gt, _, _), state = run_cycle(
            max_pcn, config, state, True, chromosomal_copy, growth_profile
        )
        frames += [label_cycle(lt, cycle, "loss"), label_cycle(gt, cycle, "gain")]
    return pd.concat(frames, ignore_index=True), state


def make_output_dirs(root_name):
    root = Path(root_name)
    fig_dir = root / "figures"
    source_dir = root / "source_data"
    fig_dir.mkdir(parents=True, exist_ok=True)
    source_dir.mkdir(parents=True, exist_ok=True)
    return root, fig_dir, source_dir


def save_figure(fig, stem, fig_dir):
    for suffix in ("pdf", "svg", "png"):
        fig.savefig(fig_dir / (stem + "." + suffix), bbox_inches="tight", facecolor="white")


def export_table(table, filename, source_dir):
    table.to_csv(source_dir / filename, index=False)


def core_metadata(figure_id):
    return pd.DataFrame([dict(
        figure=figure_id,
        core_version=CORE_VERSION,
        run_mode=RUN_MODE,
        python_version=platform.python_version(),
        max_step_hours=MAX_STEP_HOURS,
        output_step_hours=OUTPUT_STEP_HOURS,
        atol=ATOL,
        rtol=RTOL,
        negative_state_tolerance=NEGATIVE_STATE_TOLERANCE,
        composition_convergence_tolerance=COMPOSITION_CONVERGENCE_TOLERANCE,
        boundary_density_log_convergence_tolerance=BOUNDARY_DENSITY_LOG_CONVERGENCE_TOLERANCE,
        endpoint_TCN_log_convergence_tolerance=ENDPOINT_TCN_LOG_CONVERGENCE_TOLERANCE,
        consecutive_converged_cycles=CONSECUTIVE_CONVERGED_CYCLES,
        minimum_convergence_cycles=MIN_CONVERGENCE_CYCLES,
        maximum_convergence_cycles=MAX_CONVERGENCE_CYCLES,
        additional_cycles_after_convergence=ADDITIONAL_CYCLES_AFTER_CONVERGENCE,
        survival_threshold=TOTAL_DENSITY_SURVIVAL_THRESHOLD,
    )])
